# Document Extraction Engine: Kaggle demo

Upload a document and the engine will:

1. **Split** it into page images at 200 DPI.
2. **Classify** the document from page 1: category (structured, unstructured or handwritten), document type, language and rotation.
3. **Extract** the fields from each page with the model for that category.
4. **Draw** a box on each value, and build one JSON result.

**Before you start:** in the right panel, open **Session options** and set **Internet: ON** and **Accelerator: GPU T4 x2**. Then run the cells in order. Full instructions are in `docs/kaggle/README.md`.

## 1. Get the code

Clones the repository the first time, or pulls the latest version after that. Check the printed commit matches the latest on GitHub.

In [ ]:
# 1. Get the latest code: clone the first time, pull after. Needs Internet ON in the session options.
import os, subprocess
REPO = "/kaggle/working/DOC_Extractor_QUAPT"
if os.path.isdir(REPO + "/.git"):
    print(subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True).stdout)
else:
    r = subprocess.run(["git", "clone", "-b", "feature/prd-v1.3-qwen3-vl-multimodal",
                        "https://github.com/dharicharan/DOC_Extractor_QUAPT.git", REPO],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        raise SystemExit("Clone failed: turn Internet ON in the Kaggle session options and re-run.")
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 2. Install dependencies

Installs the Python packages the app needs. Only needed once per session. Takes a few minutes.

In [ ]:
# 2. Install dependencies (needed once per session; safe to re-run)
!pip install -q uv
%cd /kaggle/working/DOC_Extractor_QUAPT
!uv pip install --system -r requirements.txt
!uv pip install --system -e .
!uv pip install --system qwen-vl-utils transformers accelerate bitsandbytes gradio
print("Setup complete")

## 3. Download models (optional, but recommended)

Downloads the model weights so the first click is faster:

- **Qwen3-VL-4B-Instruct** (required): classifies and reads handwriting. About 9 GB.
- **GLM-OCR**, **Qwen3-4B-Instruct-2507**, **Qwen2.5-7B-Instruct** (optional): used for digital structured and unstructured documents. A failed download is skipped, and the app falls back to Qwen3-VL.

Expect 10 to 20 minutes on the first run.

In [ ]:
# 3. (Optional) Pre-download model weights so the first Extract click is fast.
# Weights are only downloaded here, not loaded onto the GPU: app.py loads them itself.
#  - Qwen3-VL-4B-Instruct: router + default extractor (required)
#  - GLM-OCR + a pure-text model: experimental "GLM-OCR + text LLM" engines (optional)
# If the GLM-OCR engine errors on load, run:  !uv pip install --system -U transformers  then restart and re-run.
from huggingface_hub import snapshot_download

MODELS = [
    ("Qwen/Qwen3-VL-4B-Instruct", True),
    ("zai-org/GLM-OCR", False),
    ("Qwen/Qwen3-4B-Instruct-2507", False),
    ("Qwen/Qwen2.5-7B-Instruct", False),
]
for model_id, required in MODELS:
    try:
        print(f"Downloading {model_id} ...")
        snapshot_download(model_id)
        print("  cached.")
    except Exception as e:
        msg = f"Download failed for {model_id}: {type(e).__name__}. Turn Internet ON in the session options and select a GPU."
        if required:
            raise SystemExit(msg)
        print("  skipped (optional):", msg)

## 4. Launch the app

Starts the Gradio app and prints a public link, for example `https://....gradio.live`. Open it in your browser.

The models load at startup, which takes 1 to 2 minutes. Keep this cell running while you use the app. To stop, press the stop button on the cell.

Settings such as `DOC_MAX_PAGES` and `DOC_MAX_IMAGE_PIXELS` can be set with `%env` above the launch line. See the table in `docs/kaggle/README.md`.

In [ ]:
# 4. Launch the Gradio app (public share link). Needs a GPU in the session options.
!fuser -k 7860/tcp 2>/dev/null || true
%cd /kaggle/working/DOC_Extractor_QUAPT/src/doc_extractor
!python app.py

## Using the app

1. Upload a PDF, JPG, PNG or TIFF.
2. Leave the overrides on **Auto-Detect** to classify automatically.
3. Click **Classify & Extract** and watch the **Pipeline steps** panel.
4. Read the **Visual Grounding** tab (boxes on each page) and the **Canonical JSON Output** tab (download available).

**Expected behaviour**
- The first classification is slower, because the model loads then.
- A PDF is read up to `DOC_MAX_PAGES` pages (default 5). Longer files are flagged.
- Values from the digital route have no boxes and are marked for review.

## Troubleshooting

| Problem | Fix |
| --- | --- |
| `Temporary failure in name resolution` | Turn Internet ON, restart the session, run from cell 1. |
| `No such file or directory` in cell 4 | Cell 1 did not clone. Turn Internet ON and re-run cell 1. |
| `CUDA out of memory` | Set `%env DOC_MAX_IMAGE_PIXELS=1000000` above the launch line, restart, re-run cell 4. |
| JSON error on a digital page | Pull the latest code and re-run cells 1 and 4. |
| GLM-OCR fails to load | Run `!uv pip install --system -U transformers`, restart, re-run cells 2 to 4. |

Send the steps table, the status line and the cell 1 output when asking for help.